In [33]:
from models.model import *
from models.ner_models import *
from data_access.DataProvider import *
from textpreprocessing.TextPreprocessing import *

provider = DataProvider()
preprocesor = PolishTextPreprocessor()
embedding_model = SlovenianBertModel()
rows = provider.get_articles('PL',300)
ner = PolishNER()
n = len(rows)

corpus = [row[0] for row in rows[0:n]] + [row[1] for row in rows[0:n]]
results = [row[2] for row in rows[0:n]]

In [25]:
bert_res = []
for i in range(n):
    bert_res.append(embedding_model.Transform(corpus[i],corpus[i + n]))

In [26]:
for i in range(len(corpus)):
    corpus[i] = preprocesor.preprocess_text(corpus[i], apply_lemmatization= True)
    corpus[i] = ''.join(corpus[i])

In [27]:

ner_results = []
a = []
for i in range(len(corpus)):
    ner_results.append(ner.FindNamedEntites(corpus[i]))
    a.append(([inner_list[0] for inner_list in ner_results[i]]))

In [28]:
language_model = spacy.load('hr_core_news_lg')
lemated = []
for i in range(len(a)):
    b = []
    for j in range(len(a[i])):
        doc = language_model(a[i][j])
        b.append(' '.join([token.lemma_ for token in doc]))
    lemated.append(b)
    

In [29]:
tf = TfIdfModel()

tfidf_result = []
for i in range(n):
    tfidf_matrix = tf.Transform([corpus[i], corpus[i + n]],1,1)
    tfidf_result.append(tfidf_matrix[0][1])

In [34]:
import openpyxl
from utils.utils import count_jaccard_similarity
# Create a new Excel workbook
workbook = openpyxl.Workbook()

# Select the active sheet (default is the first sheet)
sheet = workbook.active

# Write headers (optional)
headers = ['text1', 'text2', 'lexical','semantic','entities1', 'entities2', 'entitysimilarity']
sheet.append(headers)
data = []

for i in range(n):
    data.append([corpus[i], corpus[i + n], tfidf_result[i], bert_res[i], ', '.join(lemated[i]), ', '.join(lemated[i + n]), count_jaccard_similarity(lemated[i],lemated[i + n])])


for row in data:
    sheet.append(row)

# Save the workbook to a file
workbook.save('results_PL.xlsx')


In [35]:
from utils.utils import *
calculate_stats(bert_res)

Maximum: 1.0000001192092896
Minimum: -0.08464320003986359
Average: 0.54
Median: 0.55


(1.0000001192092896,
 -0.08464320003986359,
 0.5414790782704949,
 0.5482836961746216)